
# 05 — Paper figures

Regenerates every figure from `experiments/results/` only. No model is fitted here,
so the figures cannot silently disagree with the tables they are drawn from — and the
paper regenerates on a clean runtime without refitting anything.

Run the experiment scripts first:

```bash
python experiments/run_audit.py
python experiments/run_benchmark.py
python experiments/run_allocation.py
python experiments/run_horizon.py
python experiments/run_coverage_gate.py
```

In [ ]:
# --- Colab bootstrap -------------------------------------------------------
# Works in Colab and locally. In Colab, clone the repo first:
#     !git clone <repo-url> bwalloc && %cd bwalloc
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "src" / "bwalloc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

try:
    import xgboost  # noqa: F401
except ImportError:
    !pip install -q xgboost

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import bwalloc as bw
from bwalloc.plots import use_paper_style

bw.set_seed()
use_paper_style()
pd.set_option("display.width", 200)
RESULTS = ROOT / "experiments" / "results"
FIGURES = ROOT / "paper" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("bwalloc", bw.__version__, "| results:", RESULTS)

In [ ]:

from bwalloc.data import load, sampling_profile
from bwalloc.plots import (
    plot_benchmark, plot_coverage_by_group, plot_flag_report, plot_pareto,
)

profiles = {op: sampling_profile(load(op)) for op in ("gp", "robi")}
written = []

def save(fig, name):
    path = FIGURES / name
    fig.savefig(path, dpi=200, bbox_inches="tight")
    written.append(name)
    plt.close(fig)

### Figure 1 — sampling rate and autocorrelation

In [ ]:

from bwalloc.plots import plot_autocorrelation_by_lag

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, op in zip(axes, ("gp", "robi")):
    acf = pd.read_csv(RESULTS / f"audit_acf_{op}.csv")
    plot_autocorrelation_by_lag(acf, profiles[op], ax=ax)
    ax.set_title(f"{op.upper()} — {profiles[op].median_gap_min:.0f} min/sample")
fig.tight_layout()
save(fig, "fig1_autocorrelation.png")

### Figure 2 — corrected benchmark with baselines

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, op in zip(axes, ("gp", "robi")):
    plot_benchmark(pd.read_csv(RESULTS / f"benchmark_{op}_summary.csv"), ax=ax)
    ax.set_title(op.upper())
fig.tight_layout()
save(fig, "fig2_benchmark.png")

### Figure 3 — capacity–risk frontier

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, op in zip(axes, ("gp", "robi")):
    plot_pareto(pd.read_csv(RESULTS / f"pareto_{op}.csv"), ax=ax)
    ax.set_title(op.upper())
fig.tight_layout()
save(fig, "fig3_pareto.png")

### Figure 4 — the context-conditional result (GP) and its null (Robi)

In [ ]:

# One legend for the grid, and one y-range: with per-panel autoscaling the bars are
# not comparable across panels even when the axes are nominally shared.
fig, axes = plt.subplots(2, 3, figsize=(13, 7.6), sharey=True)
for r, op in enumerate(("gp", "robi")):
    per_fold = pd.read_csv(RESULTS / f"allocation_{op}_perfold.csv").fillna({"error": ""})
    for c, tau in enumerate((0.80, 0.90, 0.95)):
        ax = axes[r][c]
        plot_coverage_by_group(per_fold, tau=tau, ax=ax,
                               legend=(r == 1 and c == 1), ylim=(0.60, 1.0))
        ax.set_title(f"{op.upper()}  tau = {tau:.2f}")
        if c:
            ax.set_ylabel("")
fig.tight_layout()
save(fig, "fig4_coverage_by_group.png")

### Figure 5 — accuracy versus lead time

The multi-horizon result: the learned models stay flat while the naive forecaster collapses, so the gap between them — the value of learning — widens with lead time.

In [ ]:

from bwalloc.plots import plot_horizon

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, op in zip(axes, ("gp", "robi")):
    plot_horizon(pd.read_csv(RESULTS / f"horizon_{op}.csv"), ax=ax)
    ax.set_title(op.upper())
fig.tight_layout()
save(fig, "fig5_horizon.png")

### Figure 6 — context-flag audit

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, op in zip(axes, ("gp", "robi")):
    plot_flag_report(pd.read_csv(RESULTS / f"audit_flags_{op}.csv"), ax=ax)
    ax.set_title(op.upper())
fig.tight_layout()
save(fig, "fig6_flag_audit.png")


### Figure 7 — provisioning cost, the headline claim

Bars are the cost at the level the theory prescribes from the cost ratio
(τ* = κ/(1+κ)), so no test-set information enters the choice. The dashed line is the
best the fixed-margin heuristic can do *with* hindsight about which margin hit which
violation rate — the comparison is deliberately biased against the bars.

Blue beats that line; red does not. Context-adaptive calibration is blue on both
operators and marginal calibration is red on both, which is the whole argument.

In [ ]:

from bwalloc.plots import plot_cost

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for ax, op in zip(axes, ("gp", "robi")):
    plot_cost(pd.read_csv(RESULTS / f"cost_{op}.csv"), ax=ax)
    ax.set_title(op.upper())
fig.tight_layout()
save(fig, "fig7_cost.png")

### Figure 8 — cold-start transfer

In [ ]:

from bwalloc.plots import plot_transfer

transfer = pd.read_csv(RESULTS / "transfer_robi_to_gp.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, hours in zip(axes, sorted(transfer["horizon_hours"].unique())):
    plot_transfer(transfer, horizon_hours=hours, ax=ax)
    ax.set_title(f"{hours:g}-hour horizon")
fig.tight_layout()
save(fig, "fig8_transfer.png")

In [ ]:

print(f"wrote {len(written)} figures to {FIGURES}")
for name in written:
    print("  ", name)